# Deep learning as a feature extractor for images

_Feature Engineering (Zheng & Casari)_

**Let a convolutional network LEARN the image features, then reuse its activations instead of hand-designing SIFT or HOG.**

Every feature-engineering lesson so far hand-builds features: you decide to take counts,
       a log, a bin, a tf-idf (term frequency–inverse document frequency) weight. For images,
       the classical version of that is hand-designing descriptors like SIFT (Scale-Invariant
       Feature Transform) or HOG (Histogram of Oriented Gradients) — a human writes the
       recipe that turns pixels into a feature vector.

       Zheng & Casari close the book with the modern alternative: deep learning as the
       latest image feature-extraction technique. Instead of a human writing the recipe, a
       Convolutional Neural Network (CNN) learns the recipe from data. The key reframing
       is this: a trained image classifier is really two parts stacked together — a featurizer
       (everything up to the last hidden layer) that turns pixels into a feature vector, and a tiny
       classifier head (the final layer) that maps that vector to labels.

---

This notebook is a practice scaffold for the **Deep learning as a feature extractor for images** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch + torchvision

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# --- 1. Load a CNN pretrained on ImageNet (the expensive part, done for us) ---
resnet = models.resnet18(pretrained=True)
resnet.eval()                       # inference mode: no dropout/batchnorm updates

# --- 2. Turn it into a FEATURIZER: chop off the final classification layer ---
# resnet.fc is the last fully-connected (classifier) layer -> 1000 ImageNet classes.
# Replace it with Identity so the network now outputs the PENULTIMATE features (512-dim).
resnet.fc = nn.Identity()

# Freeze the weights -- we reuse them, we do not train them.
for p in resnet.parameters():
    p.requires_grad = False

# Standard ImageNet preprocessing the pretrained model expects.
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def deep_features(image_paths):
    """Extract the 512-dim penultimate-layer feature vector for each image."""
    feats = []
    with torch.no_grad():
        for path in image_paths:
            img = Image.open(path).convert('RGB')
            x = preprocess(img).unsqueeze(0)   # add batch dimension
            z = resnet(x)                      # z: penultimate-layer features, shape (1, 512)
            feats.append(z.squeeze(0).numpy())
    return feats

# --- 3. Use the learned features in a simple downstream classifier (transfer learning) ---
# X is a list of 512-dim vectors; train a plain logistic regression on YOUR labels.
from sklearn.linear_model import LogisticRegression
X = deep_features(my_image_paths)          # learned features, not hand-crafted SIFT/HOG
clf = LogisticRegression(max_iter=1000)
clf.fit(X, my_labels)                       # only THIS small head is trained
# predict on a new image: clf.predict(deep_features([new_path]))

## Visualize the data & results

_Learned features should be more label-efficient than raw pixels. On the digits dataset, how does downstream accuracy compare when the classifier sees RAW pixels vs a learned-feature stand-in, as we vary how many labels per class it gets to train on?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Real bundled dataset: 1797 handwritten digit images, 8x8 = 64 pixels each.
X, y = load_digits(return_X_y=True)

# A PRETRAIN pool (to LEARN the featurizer) kept disjoint from the downstream data,
# mirroring how a CNN is pretrained on a separate, larger dataset (ImageNet).
X_pre, X_rest, y_pre, y_rest = train_test_split(
    X, y, test_size=0.7, stratify=y, random_state=0)
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_rest, y_rest, test_size=0.28, stratify=y_rest, random_state=1)

scaler = StandardScaler().fit(X_pre)
sc = scaler.transform

# LEARNED featurizer: a small network trained on the pretrain pool; use its 32-unit
# hidden layer as the feature vector -- the proxy for a pretrained CNN's penultimate layer.
net = MLPClassifier(hidden_layer_sizes=(32,), max_iter=600,
                    random_state=0).fit(sc(X_pre), y_pre)
def feat(a):                                  # hidden-layer activations (ReLU)
    return np.maximum(sc(a) @ net.coefs_[0] + net.intercepts_[0], 0)

rng = np.random.RandomState(0)
for k in [2, 5, 10, 20, 40, 80]:              # labeled examples per class
    idx = np.hstack([rng.choice(np.where(y_tr_full == c)[0], k, replace=False)
                     for c in np.unique(y_tr_full)])
    for name, tr, te in [("raw pixels", sc(X_tr_full), sc(X_te)),
                         ("learned feat", feat(X_tr_full), feat(X_te))]:
        clf = LogisticRegression(max_iter=2000).fit(tr[idx], y_tr_full[idx])
        print(f"k={k:>2}  {name:<12}: acc={clf.score(te, y_te):.3f}")
# Learned features win at every k, and by the most when labels are scarce.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have 600 labeled photos for a new 3-class image task and a single laptop GPU. Why is training a ResNet from scratch a bad idea, and what does Chapter 8 tell you to do instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count what training from scratch must fit: ~11 million ResNet-18 weights. — _Fitting that many parameters needs ImageNet-scale data (~1.2M images) and serious compute — 600 images would massively overfit._
- Split the network into featurizer $f_\theta$ (up to the last hidden layer) and head $g$ (final layer). — _Only the head is tied to the original labels; the featurizer's edges/textures/parts are generic and reusable._
- Freeze a pretrained $f_\theta$, extract the penultimate vector $\mathbf{z}$ for each photo, and train a simple $g$ on those 600 vectors. — _Now you only fit a small head, which 600 examples can support, while reusing all the pretrained visual know-how._

**Answer:** Training from scratch would fit ~11 million weights on only 600 images — guaranteed overfitting and far too little data/compute. Instead use transfer learning: take a pretrained CNN, freeze it as a featurizer $f_\theta$, read off the penultimate-layer features $\mathbf{z}=f_\theta(\mathbf{I})$, and train only a simple downstream classifier $g$ on those vectors. You reuse ImageNet's learned features for free and train almost nothing yourself.

</details>

**Problem 2.** When using a pretrained CNN as a feature extractor, why do you take the activations of the penultimate (last hidden) layer rather than the input pixels, an early conv layer, or the final softmax layer?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reject raw pixels and the earliest conv layer. — _Pixels are too high-dimensional and low-level; the first conv layer only fires on edges and color blobs — too generic and low-level to classify objects directly._
- Reject the final classification (softmax) layer. — _Its outputs are probabilities over the ORIGINAL ImageNet classes — over-specialized to the pretraining labels, not your task._
- Choose the penultimate layer. — _By the last hidden layer the hierarchy has built high-level, object-relevant features that are still generic enough to transfer — the sweet spot._

**Answer:** The penultimate layer sits at the top of the learned feature hierarchy — its units respond to high-level, object-level patterns (parts, shapes) — yet it is not tied to the original labels the way the final softmax layer is. Earlier layers (and raw pixels) are too low-level; the final layer is too task-specific. So the last hidden layer's activations $\mathbf{z}$ are the most useful general-purpose feature vector.

</details>

**Problem 3.** Your team applies an ImageNet-pretrained backbone to grayscale microscope images and accuracy is disappointing. Name the likely cause from the book's pitfalls and two fixes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the pretraining data to the new data. — _ImageNet is everyday color photos; microscope images look nothing like them — this is domain shift, a pitfall the book flags._
- Consider fine-tuning instead of freezing. — _Unfreezing some layers lets the generic features adapt toward the new domain, if you have enough in-domain labels._
- Consider a closer backbone or a stronger baseline check. — _A backbone pretrained on medical/scientific images starts closer to the target; always verify the deep features beat a simple baseline._

**Answer:** The likely cause is domain shift: features learned on natural ImageNet photos transfer poorly to microscope images. Two fixes: (1) fine-tune the backbone on in-domain data so its features adapt to the new domain, and (2) start from a backbone pretrained on a closer domain (or at least confirm the deep features actually beat a simpler baseline before trusting them).

</details>